# Data Flywheel — Results

Track improvement across flywheel cycles, visualise promotion history, and understand how the flywheel drives continuous improvement.

**Prerequisites:** Multiple completed flywheel runs with at least one promoted model.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pymongo import MongoClient
from dotenv import load_dotenv
from datetime import datetime, timezone

load_dotenv()

MONGO_URI = os.getenv('MONGO_URI', 'mongodb://localhost:27017')
MONGO_DB = os.getenv('MONGO_DB', 'data_flywheel')

mongo = MongoClient(MONGO_URI)
db = mongo[MONGO_DB]
print('Connected')

## 1. Flywheel Run History

In [ ]:
runs = list(db.flywheel_runs.find({}).sort('started_at', 1))
df_runs = pd.DataFrame(runs)
df_runs['_id'] = df_runs['_id'].astype(str)

print(f'Total runs: {len(df_runs)}')
print(f'Status counts:\n{df_runs["status"].value_counts()}')
df_runs[['_id', 'status', 'started_at', 'updated_at']].tail(10)

In [ ]:
# Run status timeline
status_colors = {
    'completed': '#2ecc71',
    'failed': '#e74c3c',
    'pending': '#f39c12',
    'resuming': '#3498db',
}

fig, ax = plt.subplots(figsize=(12, 4))
for i, row in df_runs.iterrows():
    color = status_colors.get(row['status'], 'gray')
    ax.barh(0, 1, left=i, color=color, height=0.5, edgecolor='white')

patches = [plt.Rectangle((0, 0), 1, 1, color=c, label=s) for s, c in status_colors.items()]
ax.legend(handles=patches, loc='upper right')
ax.set_xlabel('Run index')
ax.set_title('Flywheel Run History')
ax.set_yticks([])
plt.tight_layout()
plt.show()

## 2. Accuracy Improvement Across Cycles

In [ ]:
experiments = list(db.experiments.find({'status': 'completed'}))
df_exp = pd.DataFrame(experiments)

if len(df_exp) > 0:
    metrics_df = pd.json_normalize(df_exp['metrics'])
    df_exp = pd.concat([df_exp.drop(columns=['metrics']), metrics_df], axis=1)

    # Join with run info to get cycle number
    run_map = {str(r['_id']): i+1 for i, r in enumerate(runs)}
    df_exp['cycle'] = df_exp['run_id'].map(run_map)

    # Plot accuracy per cycle
    cycle_accuracy = df_exp.groupby('cycle')['accuracy'].mean()

    plt.figure(figsize=(10, 5))
    plt.plot(cycle_accuracy.index, cycle_accuracy.values, 'o-', color='#2ecc71', linewidth=2, markersize=8)
    plt.axhline(0.85, color='red', linestyle='--', label='Promotion threshold')
    plt.title('Mean Accuracy Across Flywheel Cycles')
    plt.xlabel('Cycle')
    plt.ylabel('Mean Accuracy (Win Rate)')
    plt.ylim(0, 1.1)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No completed experiments found. Run make run-flywheel first.')

## 3. Promotion History

In [ ]:
# Find promoted models
try:
    model_registry = list(db.model_registry.find({}).sort('promoted_at', 1))
    if model_registry:
        df_registry = pd.DataFrame(model_registry)
        print(f'Promoted models: {len(df_registry)}')
        df_registry[['model_id', 'model_name', 'accuracy', 'latency_p95_ms', 'promoted_at']]
    else:
        print('No promotions yet.')
except Exception as e:
    print(f'Registry not available: {e}')

## 4. Cost Over Time

In [ ]:
if len(df_exp) > 0 and 'total_cost_usd' in df_exp.columns:
    cost_by_cycle = df_exp.groupby('cycle')['cost_per_1k_tokens_usd'].mean()

    plt.figure(figsize=(10, 4))
    plt.bar(cost_by_cycle.index, cost_by_cycle.values, color='#3498db', edgecolor='white')
    plt.axhline(0.02, color='red', linestyle='--', label='Max cost threshold ($0.02)')
    plt.title('Mean Cost per 1k Tokens Across Cycles')
    plt.xlabel('Cycle')
    plt.ylabel('Cost per 1k tokens (USD)')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No cost data available yet.')

## 5. Curation Funnel

In [ ]:
completed_runs = [r for r in runs if r.get('status') == 'completed']

if completed_runs:
    curation_data = []
    for i, run in enumerate(completed_runs):
        stages = run.get('stages', {})
        curation = stages.get('curation', {})
        if curation.get('status') == 'completed':
            curation_data.append({
                'cycle': i + 1,
                'logs_pulled': curation.get('logs_pulled', 0),
                'after_filter': curation.get('samples_after_filter', 0),
                'after_dedup': curation.get('samples_after_dedup', 0),
            })

    if curation_data:
        df_curation = pd.DataFrame(curation_data)

        x = range(len(df_curation))
        width = 0.25

        fig, ax = plt.subplots(figsize=(12, 5))
        ax.bar([i - width for i in x], df_curation['logs_pulled'], width, label='Raw logs', color='#e74c3c', alpha=0.8)
        ax.bar(x, df_curation['after_filter'], width, label='After filter', color='#f39c12', alpha=0.8)
        ax.bar([i + width for i in x], df_curation['after_dedup'], width, label='After dedup', color='#2ecc71', alpha=0.8)

        ax.set_xlabel('Cycle')
        ax.set_ylabel('Sample count')
        ax.set_title('Curation Funnel Across Cycles')
        ax.set_xticks(list(x))
        ax.set_xticklabels(df_curation['cycle'])
        ax.legend()
        plt.tight_layout()
        plt.show()

        print(df_curation)
else:
    print('No completed runs found.')